# R05 · Figures 4–8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R05_Figures.ipynb)

Draws the five data figures from `results/figure_source_data/` (run R01–R04 first) and saves PNG (300 dpi) and PDF files under `results/figures/`, plus journal-numbered PDFs in `results/figures/manuscript_numbering/`.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Draw the five data figures of the manuscript from results/figure_source_data/.

Run the four reproduce_*.py scripts first (they write the source-data CSVs), or use
scripts/reproduce_all.py. Each figure is written as PNG (300 dpi) and PDF under results/figures/, with
the thesis-chapter file names, and again as PDF under results/figures/manuscript_numbering/ with the
journal numbering:

  Figure_3_1_rows_vs_fingerprints          -> manuscript Fig. 4
  Figure_3_3_fp_rate_by_multiplicity_band  -> manuscript Fig. 5
  Figure_3_2_benign_transfer_progression   -> manuscript Fig. 6
  Figure_3_4_quantile_transfer             -> manuscript Fig. 7
  Figure_3_5_paired_metric_differences     -> manuscript Fig. 8

Figs. 1-3 of the manuscript are TikZ schematics (manuscript/figures/fig_m*.tex), not data figures.

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

## Reproduction: `scripts/reproduce_figures.py`

In [ ]:
import sys
from pathlib import Path


import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402


SRC = RESULTS / "figure_source_data"
OUT = RESULTS / "figures"
MAN = OUT / "manuscript_numbering"
OUT.mkdir(parents=True, exist_ok=True)
MAN.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.size": 7.5, "axes.linewidth": 0.6, "xtick.major.width": 0.6,
                     "ytick.major.width": 0.6, "pdf.fonttype": 42, "ps.fonttype": 42, "axes.spines.top": False,
                     "axes.spines.right": False, "svg.hashsalt": "nids", "pdf.compression": 6})
MM = 1 / 25.4
W1, W2 = 88 * MM, 180 * MM
LIGHT, DARK, ACC = "#b8c7d9", "#2b4a6f", "#b5462f"


def save(fig, name, manuscript_number):
    # bbox_inches="tight" guarantees that no axis label is clipped at the figure edge
    kw = dict(bbox_inches="tight", pad_inches=0.03)
    for ext in ("png", "pdf"):
        fig.savefig(OUT / f"{name}.{ext}", dpi=300, metadata={"CreationDate": None} if ext == "pdf" else None, **kw)
    fig.savefig(MAN / f"Figure_{manuscript_number}.pdf", metadata={"CreationDate": None}, **kw)
    plt.close(fig)
    print(f"wrote {name}.png/.pdf and manuscript_numbering/Figure_{manuscript_number}.pdf")


def need(name):
    p = SRC / name
    if not p.exists():
        sys.exit(f"{p} missing - run scripts/reproduce_all.py (or the reproduce_*.py scripts) first")
    return pd.read_csv(p)

### manuscript Fig. 4: rows versus model-view fingerprint groups

In [ ]:
d = need("Figure_3_1_data.csv")
fig, ax = plt.subplots(figsize=(W1, 52 * MM))
x = np.arange(len(d))
w = 0.36
ax.bar(x - w / 2, d.rows / 1e6, w, color=LIGHT, edgecolor=DARK, lw=0.5, label="Rows")
ax.bar(x + w / 2, d.fingerprint_groups / 1e6, w, color=DARK, label="Fingerprint groups")
for i, r in d.iterrows():
    ax.text(i, r.rows / 1e6 + 0.06, f"{r.redundancy_pct:.2f}%", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x, [p.replace(" ", "\n", 1) for p in d.panel])
ax.set_ylabel("Count (millions)")
ax.set_ylim(0, 3.75)
ax.legend(frameon=False, loc="upper center", ncol=2, fontsize=6.5, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(pad=0.3)
save(fig, "Figure_3_1_rows_vs_fingerprints", 4)

### manuscript Fig. 5: false-positive probability by multiplicity band

In [ ]:
d = need("Figure_3_3_data.csv")
fig, ax = plt.subplots(figsize=(W1, 58 * MM))
ax.bar(range(len(d)), d.row_rate_pct, color=DARK, width=0.7)
for i, r in enumerate(d.row_rate_pct):
    ax.text(i, r + 1.5, f"{r:.3g}" if r < 1 else f"{r:.2f}", ha="center", fontsize=6.2)
ax.set_xticks(range(len(d)), [f"{b.replace('>=', '≥')}\n{n / 1000:.1f}k\n{k:,}" for b, n, k in zip(d.band, d.rows, d.groups)], fontsize=5.6)
ax.set_xlabel("Multiplicity band (lines 2 and 3: rows; fingerprint groups)", fontsize=6.6)
ax.set_ylabel("Rows above $T_{99}$ (%)")
ax.set_ylim(0, 110)
fig.tight_layout(pad=0.3)
save(fig, "Figure_3_3_fp_rate_by_multiplicity_band", 5)

### manuscript Fig. 6: development-benign exceedance by allocation regime

In [ ]:
d = need("Figure_3_2_data.csv")
val = d.development_benign_exceedance_pct.to_numpy()
cal_final = float(d.calibration_benign_exceedance_pct.iloc[2])
fig, ax = plt.subplots(figsize=(W1, 50 * MM))
ax.scatter([0], [val[0]], s=22, facecolor="white", edgecolor=DARK, zorder=3, label="Analysis record (no retained manifest)")
ax.scatter([1, 2], val[1:], s=22, color=DARK, zorder=3, label="Development-benign exceedance (manifest)")
for i, (v, lab) in enumerate(zip(val, [f"{val[0]:.2f}%", f"{val[1]:.2f}%", f"{val[2]:.4f}%"])):
    ax.annotate(lab, (i, v), textcoords="offset points", xytext=(7, 4), va="bottom", fontsize=7)
ax.scatter([2.14], [cal_final], marker="D", s=16, facecolor="white", edgecolor=ACC, zorder=4,
           label=f"Calibration-benign exceedance, v3.4 ({cal_final:.4f}%)")
ax.axhline(1, ls="--", lw=0.7, color="grey")
ax.text(-0.42, 1.12, "nominal 1%", fontsize=6.5, color="grey")
ax.set_yscale("log")
ax.set_ylim(0.5, 400)
ax.set_xlim(-0.5, 2.7)
ax.set_xticks(range(3), ["Initial\natomic", "Source-\nbalanced", "Source- and\nmultiplicity-\nbalanced (v3.4)"], fontsize=6.8)
ax.set_ylabel("Development-benign rows\nabove $T_{99}$ (%)")
ax.legend(frameon=False, fontsize=5.6, loc="center left", bbox_to_anchor=(0.0, 0.40), handletextpad=0.3)
fig.tight_layout(pad=0.3)
save(fig, "Figure_3_2_benign_transfer_progression", 6)

### manuscript Fig. 7: quantile transfer

In [ ]:
d = need("Figure_3_4_data.csv")
q = [f"{100 * v:g}" for v in d["quantile"]]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(W2, 50 * MM))
a1.plot(range(len(d)), d.calibration_benign, "o-", color=DARK, ms=3.5, lw=0.9, label="Calibration benign")
a1.plot(range(len(d)), d.development_benign, "s--", color=ACC, ms=3, lw=0.8, mfc="white", label="Development benign")
a1.set_yscale("log")
a1.set_xticks(range(len(d)), q)
a1.set_xlabel("Percentile")
a1.set_ylabel("Reconstruction error (MSE)")
a1.set_title("(a) UNSW-NB15, v3.4 diagnostic model", fontsize=7.5)
a1.legend(frameon=False, fontsize=6)
a2.plot(range(len(d)), d.dev_to_cal_ratio, "o-", color=DARK, ms=3.5, lw=0.9, label="UNSW-NB15, prospective gate (artefact)")
a2.plot(range(len(d)), d.cicids2017_retrospective_ratio_analysis_record, "s--", color=ACC, ms=3, lw=0.8, mfc="white",
        label="CICIDS2017, retrospective check (analysis record)")
a2.axhline(1, ls=":", lw=0.7, color="grey")
a2.set_ylim(0.88, 1.12)
a2.set_xticks(range(len(d)), q)
a2.set_xlabel("Percentile")
a2.set_ylabel("Development / calibration")
a2.set_title("(b) Quantile ratio", fontsize=7.5)
a2.legend(frameon=False, fontsize=6, loc="upper right")
fig.tight_layout(pad=0.4, w_pad=1.5)
save(fig, "Figure_3_4_quantile_transfer", 7)

### manuscript Fig. 8: paired DAE - AE contrasts

In [ ]:
d = need("Figure_3_5_data.csv")
titles = {"MCC": "MCC", "Recall": "Recall", "PR-AUC": "PR-AUC", "FP_per_1000_benign": "FP per 1,000 benign"}
fig, axs = plt.subplots(1, 4, figsize=(W2, 46 * MM))
for j, (m, title) in enumerate(titles.items()):
    ax = axs[j]
    for i, (ds, c) in enumerate([("CICIDS2017", ACC), ("UNSW-NB15", DARK)]):
        r = d[(d.dataset == ds) & (d.metric == m)].iloc[0]
        ax.errorbar(i, r["mean"], yerr=[[r["mean"] - r.lo], [r.hi - r["mean"]]], fmt="o", color=c, ms=4, capsize=2.5, lw=0.9)
    ax.axhline(0, ls="--", lw=0.7, color="grey")
    ax.set_xlim(-0.6, 1.6)
    ax.set_xticks([0, 1], ["CICIDS\n2017", "UNSW-\nNB15"])
    ax.set_title(title, fontsize=7.5)
    if j == 0:
        ax.set_ylabel("DAE − AE")
fig.tight_layout(pad=0.4, w_pad=1.2)
save(fig, "Figure_3_5_paired_metric_differences", 8)

In [ ]:
# ===== Show the figures =====
from IPython.display import Image, display
for f in ["Figure_3_1_rows_vs_fingerprints", "Figure_3_3_fp_rate_by_multiplicity_band", "Figure_3_2_benign_transfer_progression",
          "Figure_3_4_quantile_transfer", "Figure_3_5_paired_metric_differences"]:
    print(f + ".png"); display(Image(filename=str(REPO_ROOT / "results" / "figures" / (f + ".png")), width=700))